In [ ]:
# !pip install open3d

In [ ]:
#ВЫВЕСТИ 3Д МОДЕЛЬ В ОТДЕЛЬНОМ ОКНЕ
# import open3d as o3d
#
# # 1. Путь к твоему 3D-файлу (.obj, .ply, .stl, .gltf)
# mesh_path = "data_cache/obj_models/41b9b4576fd043b9bcda02fc9e9fd138.obj"  # Укажи путь к своей 3D-модели
#
# # 2. Загружаем TriangleMesh из файла
# mesh = o3d.io.read_triangle_mesh(mesh_path)
#
# # Проверяем, загрузилась ли модель
# if mesh.is_empty():
#     print("Ошибка: не удалось загрузить модель! Проверь путь к файлу.")
# else:
#     print(f"Модель успешно загружена!")
#     print(f"Количество вершин (vertices): {len(mesh.vertices)}")
#     print(f"Количество граней (triangles): {len(mesh.triangles)}")
#
# # 3. Расчёт нормалей (нужно, чтобы свет правильно отражался и объект выглядел объемным)
# mesh.compute_vertex_normals()
#
# # 4. Открываем окно визуализации
# o3d.visualization.draw_geometries(
#     [mesh],
#     window_name="Open3D - Просмотр 3D меша",
#     width=800,
#     height=600
# )


#ВЫВЕСТИ 3Д МОДЕЛЬ В ОТДЕЛЬНОМ ОКНЕ

In [ ]:
from ObjaverseDataset import ObjaverseDataset
from torch.utils.data import DataLoader

In [ ]:
# dataset = ObjaverseDataset(uids=["41b9b4576fd043b9bcda02fc9e9fd138", "1f32bc0b4dd740f49faefc1cbe0e5e45"],category="Lego", start_index=1, limit=10)
# dataloader = DataLoader(dataset, batch_size=1, shuffle=False, collate_fn=dataset.collate_fn)

In [ ]:
dataset = ObjaverseDataset(start_index=10, limit=60)
dataloader = DataLoader(dataset, batch_size=1, shuffle=False, collate_fn=dataset.collate_fn)

In [ ]:
from utils import *

In [ ]:
blacklist=['c786b97d08b94d02a1fa3b87d2e86cf1', 'efd35e7d21ac482688c294e3b6c9f74e', 'dcd33159a0864de388de3a08f55e604a']
#dcd33159a0864de388de3a08f55e604a пиздец какой то
cnt = 0
highest_checkpoint = 6
for batch in dataloader:
    print(batch["uid"])
    if (batch["uid"][0] in blacklist):
        cnt+=1
        continue
    if cnt<highest_checkpoint:
        cnt+=1
        continue

    if (cnt>highest_checkpoint):
        highest_checkpoint = cnt


    mesh = batch["mesh"][0]
    print(f"Highest checkpoint {highest_checkpoint}")
    render_6_views(mesh, f"rendered_views/{batch['uid']}")
    show_rendered_grid(f"rendered_views/{batch['uid']}")
    cnt+=1


In [ ]:
#mesh = dataset[0]["mesh"]
cnt=0
for i in range(0,50):
    model = dataset[i]
    mesh = model["mesh"]
    mesh.show()

In [ ]:
model = dataset[4]
mesh = model["mesh"]
mesh.show()

In [ ]:
# !pip install pyrender

May not work out of the box
Please use
`conda install -c conda-forge open3d`
or
`brew install libusb`

In [ ]:
import os
import numpy as np
import open3d as o3d
import trimesh
def remove_platform_only(tmesh):
    """Удаляет ТОЛЬКО плоскую подставку, сохраняя голову, усы, глаза и тело."""
    # Если на входе сцена, собираем все меши
    if isinstance(tmesh, trimesh.Scene):
        geoms = [g for g in tmesh.geometry.values() if isinstance(g, trimesh.Trimesh)]
        if not geoms:
            return tmesh
        components = []
        for g in geoms:
            components.extend(g.split(only_watertight=False))
    elif isinstance(tmesh, trimesh.Trimesh):
        components = tmesh.split(only_watertight=False)
    else:
        return tmesh

    if len(components) <= 1:
        return tmesh

    cat_parts = []
    for comp in components:
        extents = comp.extents
        # Платформа характеризуется почти нулевой высотой по Y и малым числом полигонов
        is_platform = (extents[1] < 0.05 * max(extents[0], extents[2])) and (len(comp.faces) < 100)

        if not is_platform:
            cat_parts.append(comp)

    # Объединяем обратно все части кота с сохранением текстур/цветов
    if cat_parts:
        return trimesh.util.concatenate(cat_parts)
    return tmesh

def convert_trimesh_to_o3d(tmesh):
    """Convert Trimesh to Open3D TriangleMesh by preserving the colors"""

    tmesh = remove_platform_only(tmesh)


    o3d_mesh = o3d.geometry.TriangleMesh()
    o3d_mesh.vertices = o3d.utility.Vector3dVector(tmesh.vertices)
    o3d_mesh.triangles = o3d.utility.Vector3iVector(tmesh.faces)

    # Color transmition
    if hasattr(tmesh.visual, 'to_color'):
        color_visual = tmesh.visual.to_color()
        colors = color_visual.vertex_colors[:, :3] / 255.0
        o3d_mesh.vertex_colors = o3d.utility.Vector3dVector(colors)
    elif hasattr(tmesh.visual, 'vertex_colors') and tmesh.visual.vertex_colors is not None:
        colors = tmesh.visual.vertex_colors[:, :3] / 255.0
        o3d_mesh.vertex_colors = o3d.utility.Vector3dVector(colors)

    o3d_mesh.compute_vertex_normals()
    return o3d_mesh

import os
import numpy as np
import open3d as o3d
import trimesh

def render_n_views(mesh_input, n_views=6, output_dir="rendered_views", width=1008, height=1008, views=None):
    """
    Generates N images evenly distributed on a sphere around the mesh.
    Returns a list of tuples formatted as [(eye_pos, file_path), ...].

    Parameters:
        - mesh_input: file path (.obj), trimesh.Trimesh, trimesh.Scene, or o3d.geometry.TriangleMesh
        - n_views: number of viewpoints to generate if views is None
        - output_dir: directory to save rendered frames
        - width, height: resolution of output images
        - views: dict of viewpoints formatted as {"name": (eye_np_array, up_np_array)}.
                 If None, n_views camera positions will be calculated using Fibonacci Sphere algorithm.
    """
    os.makedirs(output_dir, exist_ok=True)

    # Convert input object to Open3D TriangleMesh
    if isinstance(mesh_input, str):
        mesh = o3d.io.read_triangle_mesh(mesh_input)
    elif isinstance(mesh_input, trimesh.Trimesh):
        mesh = convert_trimesh_to_o3d(mesh_input)
    elif isinstance(mesh_input, trimesh.Scene):
        tmesh = mesh_input.dump(concatenate=True)
        mesh = convert_trimesh_to_o3d(tmesh)
    else:
        mesh = mesh_input

    mesh.compute_vertex_normals()

    # Automatically calculate center and camera offset distance
    bbox = mesh.get_axis_aligned_bounding_box()
    center = bbox.get_center()
    extent = bbox.get_max_extent()
    dist = extent * 2.2  # Camera offset distance

    # Generate N evenly spaced camera views using Fibonacci Sphere algorithm
    if views is None:
        views = {}
        phi = np.pi * (3.0 - np.sqrt(5.0))  # Golden angle in radians

        for i in range(n_views):
            # Calculate y-coordinate along the sphere axis from 1 to -1
            y = 1.0 - (i / float(n_views - 1)) * 2.0 if n_views > 1 else 0.0
            radius = np.sqrt(1.0 - y * y)  # Radius at height y
            theta = phi * i  # Golden angle increment

            # Calculate direction on sphere
            x = np.cos(theta) * radius
            z = np.sin(theta) * radius

            direction = np.array([x, y, z])
            eye = center + direction * dist

            # Set correct up-vector (avoid singular vector alignment near poles)
            if np.abs(y) > 0.99:
                up = np.array([0.0, 0.0, -1.0]) if y > 0 else np.array([0.0, 0.0, 1.0])
            else:
                up = np.array([0.0, 1.0, 0.0])

            views[f"view_{i + 1:03d}"] = (eye, up)

    # Setup Open3D offscreen renderer
    renderer = o3d.visualization.rendering.OffscreenRenderer(width, height)
    material = o3d.visualization.rendering.MaterialRecord()
    material.shader = "defaultLit"

    renderer.scene.add_geometry("mesh", mesh, material)
    renderer.scene.set_background(np.array([0.2, 0.2, 0.2, 1.0]))

    half_size = (extent / 2.0) * 1.2
    aspect = width / height

    rendered_results = []

    for name, (eye, up) in views.items():
        # 1. Point the camera
        renderer.scene.camera.look_at(center, eye, up)

        # 2. Set orthographic projection
        renderer.scene.camera.set_projection(
            o3d.visualization.rendering.Camera.Projection.Ortho,
            -half_size * aspect, half_size * aspect,
            -half_size, half_size,
            0.1, dist * 5.0
        )

        # Render and save image
        img = renderer.render_to_image()
        file_path = os.path.join(output_dir, f"{name}.png")
        o3d.io.write_image(file_path, img)

        # Append tuple (camera_position, file_path) to results
        rendered_results.append((eye, file_path))

    del renderer  # Free resources

    return rendered_results

# Пример запуска с объектом из твоего датасета:
#mesh = dataset[1]["mesh"]
render_n_views(mesh, output_dir=f"rendered_views/{model['uid']}")

In [ ]:
import os
import matplotlib.pyplot as plt
from PIL import Image

def show_rendered_grid(output_dir="rendered_views"):
    views = ["front", "back", "left", "right", "top", "bottom"]
    fig, axes = plt.subplots(2, 3, figsize=(12, 8))
    axes = axes.flatten()

    for i, view in enumerate(views):
        path = os.path.join(output_dir, f"{view}.png")
        if os.path.exists(path):
            img = Image.open(path)
            axes[i].imshow(img)
            axes[i].set_title(view.capitalize(), fontsize=12)
        axes[i].axis('off')

    plt.tight_layout()
    plt.show()

# Отобразить картинки из папки
show_rendered_grid(f"rendered_views/{model['uid']}")